## Install libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [3]:
# Core
%pip install pymupdf pymupdf4llm --quiet
%pip install jiwer python-Levenshtein --quiet
%pip install pandas openpyxl --quiet

# Open-source extractors
%pip install docling --quiet
%pip install unstructured[all-docs] --quiet
%pip install marker-pdf --quiet

'''# Commercial (need API keys)
%pip install llama-parse --quiet
%pip install azure-ai-documentintelligence --quiet
%pip install boto3 --quiet
%pip install mistralai --quiet '''

print("Libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 93.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
marker-pdf 1.10.2 requires Pillow<11.0.0,>=10.1.0, but you have pillow 12.2.0 which is incompatible.
pdftext 0.6.3 requires pypdfium2==4.30.0, but you have pypdfium2 5.8.0 which is incompatible.
surya-ocr 0.17.1 requires pillow<11.0.0,>=10.2.0, but you have pillow 12.2.0 which is incompatible.
surya-ocr 0.17.1 requires pypdfium2==4.30.0, but you have pypdfium2 5.8.0 which is incompatible.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into accoun

## Imports and paths

In [ ]:
path = Path("/content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/docling/3_NVCA-Model-Voting-Agreement/images")

print(path.exists())
print(path.is_dir())
print(path.is_file())

True
False
True


In [ ]:
path.unlink()   # delete file
path.mkdir(parents=True, exist_ok=True)

In [ ]:
import os, io, re, csv, time, json, warnings
import pandas as pd
from pathlib import Path
from PIL import Image
import fitz
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ── PDF path ──────────────────────────────────────────────────────
pdf_path = "/content/drive/MyDrive/github_projects/PDF_Data_Extraction/corpus/1_Rapport-financier-semestriel-AFD-2025.pdf"

# ── Root output folder in Drive ───────────────────────────────────
output_base = Path("/content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs")

# ── API keys ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    LLAMA_KEY   = userdata.get("LLAMA_CLOUD_API_KEY") or ""
    MISTRAL_KEY = userdata.get("MISTRAL_API_KEY")     or ""
    AZURE_EP    = userdata.get("AZURE_DI_ENDPOINT")   or ""
    AZURE_KEY   = userdata.get("AZURE_DI_KEY")        or ""
except Exception:
    LLAMA_KEY = MISTRAL_KEY = AZURE_EP = AZURE_KEY = ""

pdf_path = Path(pdf_path)
pdf_stem = pdf_path.stem
assert pdf_path.exists(), f"PDF not found: {pdf_path}"

# ── Helper: returns the output folder for a given tool ────────────
# Structure: outputs/{tool_name}/{pdf_stem}/text|tables|images
def tool_dir(tool_name: str) -> Path:
    return output_base / tool_name / pdf_stem

def make_tool_folders(tool_name: str) -> dict:
    "Create and return all subfolders for one tool."
    base   = tool_dir(tool_name)
    folders = {
        "base"   : base,
        "text"   : base / "text",
        "tables" : base / "tables",
        "images" : base / "images",
        "scores" : base / "scores",
    }
    for p in folders.values():
        p.mkdir(parents=True, exist_ok=True)
    return folders

# ── Ground truth folder ───────────────────────────────────────────
GT_DIR = output_base / "ground_truth" / pdf_stem
GT_DIR.mkdir(parents=True, exist_ok=True)

# ── Create folders for every tool now so nothing breaks later ─────
TOOL_NAMES = [
    "pymupdf4llm", "docling", "unstructured",
    "marker", "llama_parse", "azure_di", "mistral_ocr"
]
for t in TOOL_NAMES:
    make_tool_folders(t)

# ── Quick PDF summary ─────────────────────────────────────────────
doc = fitz.open(str(pdf_path))
n_pages      = doc.page_count
total_chars  = sum(len(p.get_text()) for p in doc)
total_images = sum(len(p.get_images()) for p in doc)
doc.close()

print(f"PDF      : {pdf_path.name}")
print(f"Pages    : {n_pages}")
print(f"Chars    : {total_chars:,}")
print(f"Images   : {total_images}")
print()
print("Output folder structure created in Drive:")
print(f"  {output_base}")
for t in TOOL_NAMES:
    print(f"  └── {t}/{pdf_stem}/  (text/ tables/ images/ scores/)")

PDF      : 1_Rapport-financier-semestriel-AFD-2025.pdf
Pages    : 53
Chars    : 150,986
Images   : 2

Output folder structure created in Drive:
  /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs
  └── pymupdf4llm/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)
  └── docling/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)
  └── unstructured/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)
  └── marker/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)
  └── llama_parse/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)
  └── azure_di/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)
  └── mistral_ocr/1_Rapport-financier-semestriel-AFD-2025/  (text/ tables/ images/ scores/)


## Ground truth

In [ ]:
# ── Choose which pages to use as ground truth ─────────────────────
GT_TEXT_PAGE  = 1    # page number for text evaluation (1-based)
GT_TABLE_PAGE = 15   # page number for table evaluation (1-based)

# ── Auto-extract text from GT_TEXT_PAGE ───────────────────────────
doc   = fitz.open(str(pdf_path))
page  = doc[GT_TEXT_PAGE - 1]
pw    = page.rect.width

raw_blocks = page.get_text("blocks")   # (x0, y0, x1, y1, text, block_no, block_type)

# Two-column aware sort: detect if page has two columns
left_blocks  = [b for b in raw_blocks if b[0] < pw / 2]
right_blocks = [b for b in raw_blocks if b[0] >= pw / 2]
is_two_col   = len(left_blocks) > 2 and len(right_blocks) > 2

if is_two_col:
    sorted_blocks = (
        sorted(left_blocks,  key=lambda b: b[1]) +
        sorted(right_blocks, key=lambda b: b[1])
    )
else:
    sorted_blocks = sorted(raw_blocks, key=lambda b: (round(b[1]/20)*20, b[0]))

# Deduplicate
seen, clean_blocks = set(), []
for b in sorted_blocks:
    txt = re.sub(r"\s+", " ", b[4]).strip()
    if txt and txt.lower() not in seen:
        seen.add(txt.lower())
        clean_blocks.append(b)

gt_text_draft = "".join(b[4].strip() for b in clean_blocks if b[4].strip())
doc.close()

# Save text ground truth
gt_text_path = GT_DIR / f"gt_text_page{GT_TEXT_PAGE}.md"
gt_text_path.write_text(gt_text_draft, encoding="utf-8")

print(f"Text ground truth saved to Drive:")
print(f"  {gt_text_path}")
print(f"Preview (first 800 chars):")
print("-" * 50)
print(gt_text_draft[:800])
print("-" * 50)

Text ground truth saved to Drive:
  /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/ground_truth/4_scanned_document/gt_text_page1.md
Preview (first 800 chars):
--------------------------------------------------
EXAMPLE
--------------------------------------------------


In [ ]:
# ── Auto-extract table from GT_TABLE_PAGE ─────────────────────────
doc   = fitz.open(str(pdf_path))
page  = doc[GT_TABLE_PAGE - 1]

tables_found = page.find_tables()
doc.close()

gt_table_path = GT_DIR / f"gt_table_page{GT_TABLE_PAGE}.csv"

if tables_found:
    # Take the largest table on the page
    biggest = max(tables_found, key=lambda t: len(t.extract()) * len(t.extract()[0]) if t.extract() else 0)
    raw_rows = biggest.extract()
    gt_table_draft = [[c if c else "" for c in row] for row in raw_rows]

    with open(gt_table_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(gt_table_draft)

    print(f"Table ground truth auto-extracted ({len(gt_table_draft)} rows).")
    print(f"Saved to Drive: {gt_table_path}")
    print(f"Preview:")
    print(pd.DataFrame(gt_table_draft).to_string(index=False, header=False))

else:
    # No table detected — write a blank template
    print("No table auto-detected on this page.")
    print("Writing a blank template — fill it in manually in Google Drive.")
    gt_table_draft = [
        ["Column 1", "Column 2", "Column 3", "Column 4", "Column 5"],
        ["", "", "", "", ""],
        ["", "", "", "", ""],
    ]
    with open(gt_table_path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(gt_table_draft)
    print(f"Blank template saved to: {gt_table_path}")


IndexError: page 14 not in document

In [ ]:
# Load ground truth from Drive
def load_gt_text():
    p = GT_DIR / f"gt_text_page{GT_TEXT_PAGE}.md"
    return p.read_text(encoding="utf-8").strip() if p.exists() else ""

'''def load_gt_table():
    p = GT_DIR / f"gt_table_page{GT_TABLE_PAGE}.csv"
    if not p.exists():
        return []
    with open(p, newline="", encoding="utf-8") as f:
        return list(csv.reader(f))'''

GT_TEXT  = load_gt_text()
#GT_TABLE = load_gt_table()

print(f"Ground truth text  : {len(GT_TEXT)} chars loaded")
#print(f"Ground truth table : {len(GT_TABLE)} rows loaded")


Ground truth text  : 3802 chars loaded


## Scoring functions (CER + TEDS)

In [ ]:
from jiwer import cer as jiwer_cer
import Levenshtein

ALL_SCORES = []   # collects one row per tool

def compute_cer(extracted: str, reference: str) -> float:
    "Character Error Rate — 0.0 = perfect, lower is better."
    ref = re.sub(r"\s+", " ", reference.strip())
    hyp = re.sub(r"\s+", " ", extracted.strip())
    if not ref:
        return 0.0
    return round(jiwer_cer(ref, hyp), 4)

def compute_teds(pred: list, gt: list) -> float:
    "Table similarity — 1.0 = perfect, higher is better."
    if not gt:   return 1.0
    if not pred: return 0.0
    def to_str(tbl):
        return "(table " + "".join(
            "(tr " + "".join(f"(td {str(c).strip()})" for c in row) + ")"
            for row in tbl
        ) + ")"
    s1, s2 = to_str(pred), to_str(gt)
    return round(1 - Levenshtein.distance(s1, s2) / max(len(s1), len(s2)), 4)

def stars(val, metric):
    if metric == "cer":
        thresholds = [(0.05,5),(0.10,4),(0.20,3),(0.40,2)]
    else:
        thresholds = [(0.95,5),(0.85,4),(0.70,3),(0.50,2)]
    for t, s in thresholds:
        if (val <= t if metric=="cer" else val >= t):
            return "★" * s + "☆" * (5 - s)
    return "★☆☆☆☆"

def show_scores(tool_name, text, table, elapsed, cost_note="free"):
    GT_TEXT  = load_gt_text()    # always reload in case file was edited
    #GT_TABLE = load_gt_table()

    cer  = compute_cer(text, GT_TEXT)   if GT_TEXT  else None
    #teds = compute_teds(table, GT_TABLE) if GT_TABLE else None

    print()
    print("┌" + "─" * 53 + "┐")
    print(f"│  SCORES — {tool_name:<42}│")
    print("├" + "─" * 53 + "┤")
    if cer  is not None:
        print(f"│  CER  (text)  : {cer:.4f}   {stars(cer,'cer'):<15}        │")
    #if teds is not None:
        #print(f"│  TEDS (table) : {teds:.4f}   {stars(teds,'teds'):<15}        │")
    print(f"│  Time         : {elapsed:.1f}s{' '*38}│")
    print(f"│  Cost         : {cost_note:<37}│")
    print("└" + "─" * 53 + "┘")
    print("  CER:  0.0=perfect · <0.10=good · >0.30=poor")
    #print("  TEDS: 1.0=perfect · >0.85=good · <0.50=poor")

    row = {
        "tool"   : tool_name,
        "CER"    : cer  if cer  is not None else "N/A",
        #"TEDS"   : teds if teds is not None else "N/A",
        "time_s" : round(elapsed, 1),
        "cost"   : cost_note,
    }
    ALL_SCORES.append(row)

    # Save individual score to Drive
    score_path = SCORES_DIR / f"{tool_name.replace(' ','_')}_scores.json"
    with open(score_path, "w") as f:
        json.dump(row, f, indent=2)

print("Scoring functions ready.")
print(f"Individual score files will be saved to: {SCORES_DIR}")


Scoring functions ready.
Individual score files will be saved to: /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/scores


In [ ]:
from jiwer import cer as jiwer_cer
import Levenshtein

# ── CER ───────────────────────────────────────────────────────────
def compute_cer(extracted_text: str, reference_text: str) -> float:
    """Character Error Rate. 0.0 = perfect. Lower is better."""
    ref = re.sub(r"\s+", " ", reference_text.strip())
    hyp = re.sub(r"\s+", " ", extracted_text.strip())
    if not ref:
        return 0.0
    return round(jiwer_cer(ref, hyp), 4)

# ── TEDS ──────────────────────────────────────────────────────────
def compute_teds(pred_table: list, gt_table: list) -> float:
    """Table similarity 0.0–1.0. 1.0 = perfect match. Higher is better."""
    if not gt_table:
        return 1.0
    if not pred_table:
        return 0.0
    # Linearise tables as bracketed strings then compare edit distance
    def to_string(table):
        rows = []
        for row in table:
            cells = "".join(f"(td {str(c).strip()})" for c in row)
            rows.append(f"(tr {cells})")
        return f"(table {''.join(rows)})"
    s1 = to_string(pred_table)
    s2 = to_string(gt_table)
    dist = Levenshtein.distance(s1, s2)
    return round(1.0 - dist / max(len(s1), len(s2)), 4)

# ── Score display ─────────────────────────────────────────────────
def show_scores(tool_name, extracted_text, extracted_table, elapsed):
    cer  = compute_cer(extracted_text, GT_TEXT)
    teds = compute_teds(extracted_table, GT_TABLE)

    # CER to star rating
    if cer <= 0.05:   cer_stars  = 5.0
    elif cer <= 0.10: cer_stars  = 4.0
    elif cer <= 0.20: cer_stars  = 3.0
    elif cer <= 0.40: cer_stars  = 2.0
    else:             cer_stars  = 1.0

    # TEDS to star rating
    if teds >= 0.95:   teds_stars = 5.0
    elif teds >= 0.85: teds_stars = 4.0
    elif teds >= 0.70: teds_stars = 3.0
    elif teds >= 0.50: teds_stars = 2.0
    else:              teds_stars = 1.0

    print("=" * 55)
    print(f"  SCORES FOR: {tool_name}")
    print("=" * 55)
    print(f"  CER  (text accuracy)  : {cer:.4f}  {cer_stars}")
    print(f"  TEDS (table accuracy) : {teds:.4f}  {teds_stars}")
    print(f"  Time                  : {elapsed:.1f}s")
    print(f"  CER note : 0.0 = perfect | <0.10 = good | >0.30 = poor")
    print(f"  TEDS note: 1.0 = perfect | >0.85 = good | <0.50 = poor")
    print("=" * 55)
    return {"tool": tool_name, "CER": cer, "TEDS": teds, "time_s": round(elapsed,1)}

# Collect all results here
ALL_SCORES = []
print("Scoring functions ready.")


Scoring functions ready.


## Tool 1 · PyMuPDF4LLM
**Licence:** AGPL / commercial  
**Best for:** Fast baseline, native PDF text, no ML required  

In [ ]:
import fitz
import pymupdf4llm

TOOL = "pymupdf4llm"
dirs = make_tool_folders(TOOL)

t0  = time.perf_counter()
doc = fitz.open(str(pdf_path))

# ── Text — save each page as a separate .md file ──────────────────
full_md_text = pymupdf4llm.to_markdown(str(pdf_path))

# Split by page and save individually
page_texts = full_md_text.split("-----")   # pymupdf4llm uses ----- as page separator
for pg_i, page_text in enumerate(page_texts):
    if page_text.strip():
        out_file = dirs["text"] / f"page_{pg_i+1:03d}.md"
        out_file.write_text(page_text.strip(), encoding="utf-8")

# Also save the full document as one file
(dirs["text"] / "full_document.md").write_text(full_md_text, encoding="utf-8")

# ── Tables — save each as a separate .csv ─────────────────────────
pymupdf_tables = []
table_count = 0
for page in doc:
    for tbl in page.find_tables():
        rows = [[c if c else "" for c in row] for row in tbl.extract()]
        pymupdf_tables.append({"page": page.number + 1, "data": rows})
        csv_file = dirs["tables"] / f"page_{page.number+1:03d}_table_{table_count+1}.csv"
        with open(csv_file, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerows(rows)
        table_count += 1

# ── Images — save each image ──────────────────────────────────────
saved_imgs = []
for pg_i, page in enumerate(doc):
    for im_i, img_info in enumerate(page.get_images(full=True)):
        xref = img_info[0]
        base = doc.extract_image(xref)
        dest = dirs["images"] / f"page_{pg_i+1:03d}_img_{im_i+1}.{base['ext']}"
        dest.write_bytes(base["image"])
        saved_imgs.append(str(dest))
doc.close()

elapsed_pymupdf = time.perf_counter() - t0

print(f"Time   : {elapsed_pymupdf:.1f}s")
print(f"Pages  : {len(page_texts)} text files saved")
print(f"Tables : {table_count} CSV files saved")
print(f"Images : {len(saved_imgs)} files saved")
print()
print(f"All output in Drive:")
print(f"  Text   → {dirs['text']}")
print(f"  Tables → {dirs['tables']}")
print(f"  Images → {dirs['images']}")
print()
print("── Text preview (first 600 chars) ──")
print(full_md_text[:600])

=== Document parser messages ===
                                                                                                                                        Using Tesseract for OCR processing.

Time   : 19.1s
Pages  : 5 text files saved
Tables : 0 CSV files saved
Images : 0 files saved

All output in Drive:
  Text   → /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/pymupdf4llm/3_NVCA-Model-Voting-Agreement/text
  Tables → /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/pymupdf4llm/3_NVCA-Model-Voting-Agreement/tables
  Images → /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/pymupdf4llm/3_NVCA-Model-Voting-Agreement/images

── Text preview (first 600 chars) ──
**This sample document is the work product of a national coalition of attorneys who specialize in venture capital financings, working under the auspices of the NVCA. This document is intended to serve as a starting point only and should be tailored to meet your 

In [ ]:
# ── Table preview ─────────────────────────────────────────────────
if pymupdf_tables:
    t = pymupdf_tables[0]
    print(f"First table (page {t['page']}, {len(t['data'])} rows):")
    print(pd.DataFrame(t["data"]).to_string(index=False, header=False))
else:
    print("No tables found.")

No tables found.


In [ ]:
# ── Image preview ─────────────────────────────────────────────────
if saved_imgs:
    img = Image.open(saved_imgs[0])
    plt.figure(figsize=(6,4))
    plt.imshow(img); plt.axis("off")
    plt.title("PyMuPDF4LLM — first image")
    plt.tight_layout(); plt.show()
else:
    print("No raster images in this PDF.")

No raster images in this PDF.


In [ ]:
# ── SCORES ────────────────────────────────────────────────────────
pred_tbl = pymupdf_tables[0]["data"] if pymupdf_tables else []
show_scores(TOOL, full_md_text, pred_tbl, elapsed_pymupdf)

NameError: name 'show_scores' is not defined

## Tool 2 · Docling
**Licence:** MIT (IBM)  
**Best for:** Best table structure, heading detection, multi-column layout  

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

TOOL = "docling"
dirs = make_tool_folders(TOOL)

opts = PdfPipelineOptions()
opts.do_ocr             = True
opts.do_table_structure = True

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
)

t0     = time.perf_counter()
result = converter.convert(str(pdf_path))
doc_dl = result.document

# ── Text — save full document + one file per page ─────────────────
docling_text  = doc_dl.export_to_markdown()
docling_plain = doc_dl.export_to_text()

# Save full document
(dirs["text"] / "full_document.md").write_text(docling_text, encoding="utf-8")

# Save page by page
# Docling pages are accessible via doc_dl.pages
for pg in getattr(doc_dl, "pages", []):
    pg_num  = getattr(pg, "page_no", 0) + 1   # 0-based → 1-based
    pg_text = getattr(pg, "export_to_markdown", lambda: "")()
    if pg_text and pg_text.strip():
        (dirs["text"] / f"page_{pg_num:03d}.md").write_text(pg_text.strip(), encoding="utf-8")

# ── Tables — fix the deprecation warning by passing doc argument ──
docling_tables = []
for tbl_idx, tbl_item in enumerate(doc_dl.tables):
    try:
        # New API: pass doc argument to suppress deprecation warning
        df = tbl_item.export_to_dataframe(doc=doc_dl)
    except TypeError:
        # Older docling version — fall back to old call
        df = tbl_item.export_to_dataframe()

    rows = df.values.tolist()
    docling_tables.append(rows)

    # Save as CSV
    csv_file = dirs["tables"] / f"table_{tbl_idx+1:03d}.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(list(df.columns))   # header
        writer.writerows(rows)

# ── Figures ───────────────────────────────────────────────────────
saved_imgs = []
for i, fig in enumerate(getattr(doc_dl, "pictures", [])):
    try:
        img_bytes = fig.image.as_bytes()
        dest = dirs["images"] / f"figure_{i+1:03d}.png"
        dest.write_bytes(img_bytes)
        saved_imgs.append(str(dest))
    except Exception:
        pass

elapsed_docling = time.perf_counter() - t0

print(f"Time    : {elapsed_docling:.1f}s")
print(f"Tables  : {len(docling_tables)} CSV files saved → {dirs['tables']}")
print(f"Figures : {len(saved_imgs)} files saved → {dirs['images']}")
print(f"Text    : saved → {dirs['text']}")
print()
print("── Text preview (first 600 chars) ──")
print(docling_text[:600])

[INFO] 2026-05-31 20:30:12,801 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-31 20:30:12,822 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-31 20:30:12,824 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-31 20:30:12,957 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-31 20:30:12,962 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-31 20:30:12,963 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-31 20:30:13,024 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-31 20:30:13,062 [RapidOCR] download_file.py:60: File exists and is valid: /usr

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ── Table preview ─────────────────────────────────────────────────
if docling_tables:
    print(f"First table ({len(docling_tables[0])} rows):")
    print(pd.DataFrame(docling_tables[0]).to_string(index=False, header=False))
else:
    print("No tables extracted.")

First table (28 rows):
                                                                                                                                                                                                                                                A.            Management report............................................................................................................................4
                                                                                                   1. AFD Group activities.......................................................................................................................4                                                                                                                                                          
                                                                                                          2. Recent changes and outlook................................................

In [ ]:
# ── Image preview ─────────────────────────────────────────────────
if saved_imgs:
    img = Image.open(saved_imgs[0])
    plt.figure(figsize=(6, 4))
    plt.imshow(img); plt.axis("off")
    plt.title("Docling — first figure extracted")
    plt.tight_layout(); plt.show()
else:
    print("No figures extracted (vector charts won't appear as raster images).")

No figures extracted (vector charts won't appear as raster images).


In [ ]:
# ── SCORES ────────────────────────────────────────────────────────
pred_tbl_docling = docling_tables[0] if docling_tables else []
show_scores("Docling", docling_plain, pred_tbl_docling, elapsed_docling)



┌─────────────────────────────────────────────────────┐
│  SCORES — Docling                                   │
├─────────────────────────────────────────────────────┤
│  CER  (text)  : 39.9685   ★☆☆☆☆                  │
│  TEDS (table) : 0.1106   ★☆☆☆☆                  │
│  Time         : 188.0s                                      │
│  Cost         : free                                 │
└─────────────────────────────────────────────────────┘
  CER:  0.0=perfect · <0.10=good · >0.30=poor
  TEDS: 1.0=perfect · >0.85=good · <0.50=poor


## Tool 3 · Unstructured
**Licence:** Apache 2.0  
**Best for:** Element classification (titles, lists, tables, images), clean licence  

In [ ]:
from unstructured.partition.pdf import partition_pdf

t0       = time.perf_counter()
elements = partition_pdf(
    filename=str(pdf_path),
    strategy="fast",           # switch to "hi_res" for better accuracy
    languages=["eng", "fra"],
)

# ── Text ──────────────────────────────────────────────────────────
parts = []
for el in elements:
    txt     = str(el).strip()
    el_type = type(el).__name__
    if not txt: continue
    if el_type == "Title":        parts.append(f"## {txt}")
    elif el_type == "ListItem":   parts.append(f"- {txt}")
    else:                          parts.append(txt)

unstr_text = " ".join(parts)

# ── Tables ────────────────────────────────────────────────────────
unstr_tables = []
for el in elements:
    if type(el).__name__ == "Table":
        html = getattr(el.metadata, "text_as_html", None)
        if html:
            try:
                for df in pd.read_html(html):
                    unstr_tables.append(df.astype(str).values.tolist())
            except Exception:
                pass

# ── Images ────────────────────────────────────────────────────────
img_out = IMGS_DIR / "unstructured"
img_out.mkdir(parents=True, exist_ok=True)
saved_imgs = []
for el in elements:
    if type(el).__name__ == "Image":
        img_bytes = getattr(el.metadata, "image_bytes", None)
        if img_bytes:
            dest = img_out / f"img_{len(saved_imgs)+1}.png"
            dest.write_bytes(img_bytes)
            saved_imgs.append(str(dest))

elapsed_unstr = time.perf_counter() - t0

tool_text_dir = Path(OUTPUT_BASE) / "unstructured" / pdf_stem
tool_text_dir.mkdir(parents=True, exist_ok=True)
(tool_text_dir / "text.md").write_text(unstr_text, encoding="utf-8")

print(f"Time     : {elapsed_unstr:.1f}s")
print(f"Elements : {len(elements)}")
print(f"Tables   : {len(unstr_tables)}")
print(f"Images   : {len(saved_imgs)} saved to Drive → {img_out}")
print()
print("── Text preview ──")
print(unstr_text[:600])


Time     : 13.8s
Elements : 3177
Tables   : 0
Images   : 0 saved to Drive → /content/drive/MyDrive/github_projects/PDF_Data_Extraction/outputs/images/unstructured

── Text preview ──
## Half-year financial report 30 June 2025 AFD – 2025 half-year financial report ## Contents A. Management report ............................................................................................................................ 4 - 1. AFD Group activities ....................................................................................................................... 4 - 2. Recent changes and outlook .......................................................................................................... 6 2.1. Crises in several countries .....................................


In [ ]:
if unstr_tables:
    print(f"First table ({len(unstr_tables[0])} rows):")
    print(pd.DataFrame(unstr_tables[0]).to_string(index=False, header=False))
else:
    print("No tables extracted.")

No tables extracted.


In [ ]:
if saved_imgs:
    img = Image.open(saved_imgs[0])
    plt.figure(figsize=(6, 4))
    plt.imshow(img); plt.axis("off")
    plt.title("Unstructured — first image")
    plt.tight_layout(); plt.show()
else:
    print("No images found.")

No images found.


In [ ]:
pred_tbl_unstr = unstr_tables[0] if unstr_tables else []
show_scores("Unstructured", unstr_text, pred_tbl_unstr, elapsed_unstr)


┌─────────────────────────────────────────────────────┐
│  SCORES — Unstructured                              │
├─────────────────────────────────────────────────────┤
│  CER  (text)  : 35.5910   ★☆☆☆☆                  │
│  TEDS (table) : 0.0000   ★☆☆☆☆                  │
│  Time         : 13.8s                                      │
│  Cost         : free                                 │
└─────────────────────────────────────────────────────┘
  CER:  0.0=perfect · <0.10=good · >0.30=poor
  TEDS: 1.0=perfect · >0.85=good · <0.50=poor


## Tool 3 · Unstructured
**Licence:** Apache 2.0 | **Strength:** Clean licence, good element classification

In [ ]:
from unstructured.partition.pdf import partition_pdf

TOOL = "unstructured"
dirs = make_tool_folders(TOOL)

t0       = time.perf_counter()
elements = partition_pdf(
    filename=str(pdf_path),
    strategy="fast",
    languages=["eng", "fra"],
)

# ── Text — group by page, save each page as .md ───────────────────
pages_text = {}
for el in elements:
    txt     = str(el).strip()
    pg_num  = getattr(el.metadata, "page_number", 1) or 1
    el_type = type(el).__name__
    if not txt: continue
    if el_type == "Title":       line = f"## {txt}"
    elif el_type == "ListItem":  line = f"- {txt}"
    else:                         line = txt
    pages_text.setdefault(pg_num, []).append(line)

full_text_parts = []
for pg_num in sorted(pages_text.keys()):
    page_content = "

".join(pages_text[pg_num])
    full_text_parts.append(page_content)
    (dirs["text"] / f"page_{pg_num:03d}.md").write_text(page_content, encoding="utf-8")

unstr_text = "

".join(full_text_parts)
(dirs["text"] / "full_document.md").write_text(unstr_text, encoding="utf-8")

# ── Tables ────────────────────────────────────────────────────────
unstr_tables = []
tbl_count = 0
for el in elements:
    if type(el).__name__ == "Table":
        html = getattr(el.metadata, "text_as_html", None)
        if html:
            try:
                for df in pd.read_html(html):
                    rows = df.astype(str).values.tolist()
                    unstr_tables.append(rows)
                    pg_num  = getattr(el.metadata, "page_number", 1) or 1
                    csv_file = dirs["tables"] / f"page_{pg_num:03d}_table_{tbl_count+1}.csv"
                    with open(csv_file, "w", newline="", encoding="utf-8") as f:
                        csv.writer(f).writerows(rows)
                    tbl_count += 1
            except Exception:
                pass

# ── Images ────────────────────────────────────────────────────────
saved_imgs = []
for el in elements:
    if type(el).__name__ == "Image":
        img_bytes = getattr(el.metadata, "image_bytes", None)
        if img_bytes:
            pg_num  = getattr(el.metadata, "page_number", 1) or 1
            dest = dirs["images"] / f"page_{pg_num:03d}_img_{len(saved_imgs)+1}.png"
            dest.write_bytes(img_bytes)
            saved_imgs.append(str(dest))

elapsed_unstr = time.perf_counter() - t0

print(f"Time     : {elapsed_unstr:.1f}s")
print(f"Pages    : {len(pages_text)} text files saved → {dirs['text']}")
print(f"Tables   : {tbl_count} CSV files saved → {dirs['tables']}")
print(f"Images   : {len(saved_imgs)} files saved → {dirs['images']}")
print()
print("── Text preview (first 600 chars) ──")
print(unstr_text[:600])


In [ ]:
if unstr_tables:
    print(f"First table ({len(unstr_tables[0])} rows):")
    print(pd.DataFrame(unstr_tables[0]).to_string(index=False, header=False))
else:
    print("No tables extracted.")

In [ ]:
pred_tbl = unstr_tables[0] if unstr_tables else []
show_scores(TOOL, unstr_text, pred_tbl, elapsed_unstr)

## Tool 4 · Marker
**Licence:** GPL ⚠ | **Strength:** Very clean markdown, good on complex layouts

In [ ]:
from marker.convert import convert_single_pdf
from marker.models import load_all_models

TOOL = "marker"
dirs = make_tool_folders(TOOL)

print("Loading Marker models (downloads ~1-2GB on first run)...")
marker_models = load_all_models()
print("Models ready.")

t0 = time.perf_counter()

marker_text, images_dict, metadata = convert_single_pdf(
    str(pdf_path), marker_models,
    langs=["English", "French"],
    batch_multiplier=1,
)

# ── Text — split by page marker and save each page ────────────────
# Marker doesn't give explicit page breaks — split on double newlines between sections
pages_raw = re.split(r"
{3,}", marker_text)
for pg_i, page_content in enumerate(pages_raw):
    if page_content.strip():
        (dirs["text"] / f"page_{pg_i+1:03d}.md").write_text(page_content.strip(), encoding="utf-8")

(dirs["text"] / "full_document.md").write_text(marker_text, encoding="utf-8")

# ── Tables — parse markdown tables and save as CSV ────────────────
marker_tables = []
tbl_count = 0
for raw in re.findall(r"(\|.+\|
(?:\|[-:| ]+\|
)?(?:\|.+\|
)*)", marker_text):
    rows = []
    for line in raw.strip().splitlines():
        if "|" in line and not re.match(r"^[\|\s\-:]+$", line):
            rows.append([c.strip() for c in line.strip("|").split("|")])
    if rows:
        marker_tables.append(rows)
        csv_file = dirs["tables"] / f"table_{tbl_count+1:03d}.csv"
        with open(csv_file, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerows(rows)
        tbl_count += 1

# ── Images ────────────────────────────────────────────────────────
saved_imgs = []
for name, pil_img in images_dict.items():
    dest = dirs["images"] / name
    pil_img.save(str(dest))
    saved_imgs.append(str(dest))

elapsed_marker = time.perf_counter() - t0

print(f"Time   : {elapsed_marker:.1f}s")
print(f"Pages  : {len([p for p in pages_raw if p.strip()])} text files saved → {dirs['text']}")
print(f"Tables : {tbl_count} CSV files saved → {dirs['tables']}")
print(f"Images : {len(saved_imgs)} files saved → {dirs['images']}")
print()
print("── Text preview (first 600 chars) ──")
print(marker_text[:600])

In [ ]:
if marker_tables:
    print(f"First table ({len(marker_tables[0])} rows):")
    print(pd.DataFrame(marker_tables[0]).to_string(index=False, header=False))
else:
    print("No markdown tables found.")

In [ ]:
if saved_imgs:
    img = Image.open(saved_imgs[0])
    plt.figure(figsize=(6,4))
    plt.imshow(img); plt.axis("off")
    plt.title("Marker — first image"); plt.tight_layout(); plt.show()
else:
    print("No images found.")

In [ ]:
pred_tbl = marker_tables[0] if marker_tables else []
show_scores(TOOL, marker_text, pred_tbl, elapsed_marker)

## Tool 5 · LlamaParse
**Licence:** Commercial (~10k free pages/month) | **Key:** LLAMA_CLOUD_API_KEY in Secrets